In [0]:
# Run once
%pip install sentence-transformers faiss-cpu

dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql import functions as F
import pandas as pd

df = spark.table("primeinsurance.gold_layer.dim_policy")

display(df)

policy_number,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,car_id,customer_id,ingestion_timestamp,md_created_datetime
291902,2018-02-17,IL,500/1000,1000,1115.81,0,124438,167094,2026-03-26T23:57:20.618Z,2026-03-26T23:57:52.010Z
311783,2016-11-18,OH,100/300,500,1233.85,0,124086,144393,2026-03-26T23:57:20.618Z,2026-03-26T23:57:52.010Z
847123,2015-11-19,WV,100/300,500,1442.27,0,123924,137596,2026-03-26T23:57:20.618Z,2026-03-26T23:57:52.010Z
607259,2016-03-02,OH,250/500,500,1390.72,0,124294,143119,2026-03-26T23:57:20.618Z,2026-03-26T23:57:52.010Z
368050,2017-10-03,SC,500/1000,2000,1167.30,4000000,123602,106992,2026-03-26T23:57:20.618Z,2026-03-26T23:57:52.010Z
718829,2016-02-21,OH,250/500,2000,1568.47,4000000,124302,120341,2026-03-26T23:57:20.618Z,2026-03-26T23:57:52.010Z
563878,2016-07-16,SC,250/500,500,956.69,0,123828,114713,2026-03-26T23:57:20.618Z,2026-03-26T23:57:52.010Z
790225,2015-05-01,OH,250/500,500,964.92,0,124198,105676,2026-03-26T23:57:20.618Z,2026-03-26T23:57:52.010Z
808544,2015-05-02,IL,500/1000,1000,1358.91,0,124133,100524,2026-03-26T23:57:20.618Z,2026-03-26T23:57:52.010Z
900628,2016-10-22,SC,500/1000,1000,1690.27,0,123577,124303,2026-03-26T23:57:20.618Z,2026-03-26T23:57:52.010Z


In [0]:
def row_to_text(row):
    return f"""
Policy Number: {row.policy_number}
Deductible: {row.policy_deductable}
Umbrella Coverage: {row.umbrella_limit}
Policy State: {row.policy_state}
"""

def policy_docs_generator(iterator):
    for pdf in iterator:  # each pdf is a pandas DataFrame
        yield pd.DataFrame({
            "policy_doc": [
                row_to_text(row) for row in pdf.itertuples(index=False)
            ]
        })

policy_docs_df = df.mapInPandas(policy_docs_generator, schema="policy_doc string")

policy_docs = [row.policy_doc for row in policy_docs_df.collect()]

In [0]:
def chunk_text(text, chunk_size=200, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start+chunk_size])
        start += chunk_size - overlap
    return chunks

all_chunks = []
for doc in policy_docs:
    chunks = chunk_text(doc)
    all_chunks.extend(chunks)

In [0]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(all_chunks, show_progress_bar=True)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-fb2b318a-6330-43e7-87f6-eeddddd02634/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [0]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

In [0]:
def retrieve_chunks(query, k=5):
    query_embedding = model.encode([query])
    
    distances, indices = index.search(np.array(query_embedding), k)
    
    results = [all_chunks[i] for i in indices[0]]
    return results

In [0]:
import re

def extract_policy_info(text):
    policy = re.search(r"Policy Number:\s*(\S+)", text)
    umbrella = re.search(r"Umbrella Coverage:\s*(\S+)", text)
    deductible = re.search(r"Deductible:\s*(\S+)", text)
    coverage = re.search(r"Coverage Tier:\s*(\S+)", text)
    state = re.search(r"Policy State:\s*(\S+)", text)

    return {
        "policy": policy.group(1) if policy else None,
        "umbrella": umbrella.group(1) if umbrella else None,
        "deductible": deductible.group(1) if deductible else None,
        "coverage": coverage.group(1) if coverage else None,
        "state": state.group(1) if state else None
    }


def generate_answer(query, retrieved_chunks):
    query = query.lower()
    extracted = [extract_policy_info(chunk) for chunk in retrieved_chunks]

    # ---- Question Type Detection ----

    # 1. Umbrella coverage query
    if "umbrella" in query:
        policies = [e["policy"] for e in extracted if e["umbrella"] == "Yes"]
        return f"Policies with umbrella coverage: {policies}"

    # 2. Deductible lookup
    elif "deductible" in query:
        return "\n".join([
            f"Policy {e['policy']} has deductible {e['deductible']}"
            for e in extracted if e["policy"]
        ])

    # 3. Coverage comparison
    elif "compare" in query:
        return "\n".join([
            f"Policy {e['policy']} has {e['coverage']} coverage"
            for e in extracted if e["policy"]
        ])

    # 4. State-based filter
    elif "california" in query or "state" in query:
        policies = [e["policy"] for e in extracted if e["state"]]
        return f"Policies found: {policies}"

    # 5. Default fallback
    else:
        return "Relevant policies:\n" + "\n".join([
            f"Policy {e['policy']}"
            for e in extracted if e["policy"]
        ])

In [0]:
def compute_confidence(distances):
    # Lower distance = higher confidence
    return float(1 / (1 + distances[0][0]))

In [0]:
def rag_query(query):
    query_embedding = model.encode([query])
    distances, indices = index.search(np.array(query_embedding), 5)

    retrieved = [all_chunks[i] for i in indices[0]]
    answer = generate_answer(query, retrieved)
    confidence = compute_confidence(distances)

    return answer, confidence, retrieved

In [0]:
def log_query(query, answer, confidence, sources):
    log_df = spark.createDataFrame([(
        query,
        answer,
        float(confidence),
        " | ".join(sources)
    )], ["question", "answer", "confidence_score", "source_policies"])

    log_df.write.mode("append").saveAsTable("primeinsurance.gold_layer.rag_query_history")

In [0]:
queries = [
    "What is the deductible for policy 12345?",
    "Which policies have umbrella coverage?",
    "Compare basic vs premium coverage policies",
    "What policies exist in California?",
    "Which policies have high bodily injury limits?"
]

for q in queries:
    answer, confidence, sources = rag_query(q)
    
    print("Q:", q)
    print("A:", answer)
    print("Confidence:", confidence)
    print("-"*50)

    log_query(q, answer, confidence, sources)

Q: What is the deductible for policy 12345?
A: Policy 143924 has deductible 1000
Policy 143038 has deductible 500
Policy 179538 has deductible 2000
Policy 345539 has deductible 1000
Policy 352120 has deductible 500
Confidence: 0.671747624874115
--------------------------------------------------
Q: Which policies have umbrella coverage?
A: Policies with umbrella coverage: []
Confidence: 0.6202307343482971
--------------------------------------------------
Q: Compare basic vs premium coverage policies
A: Policy 641845 has None coverage
Policy 620075 has None coverage
Policy 164464 has None coverage
Policy 744557 has None coverage
Policy 149839 has None coverage
Confidence: 0.47564202547073364
--------------------------------------------------
Q: What policies exist in California?
A: Policies found: ['108270', '290162', '681486', '670142', '744557']
Confidence: 0.4678118824958801
--------------------------------------------------
Q: Which policies have high bodily injury limits?
A: Releva